# Extension: Bootstrap Confidence Interval for a Gender Difference in FSIQ

This notebook **extends Part 1** of `ai/stats_python.ipynb`, which compared intelligence-related measures between males and females in the brain-size dataset using Welch's *t*-tests and Mann–Whitney *U* tests.

**New method:** A **bootstrap confidence interval** for the difference in mean Full-Scale IQ (FSIQ) between gender groups.

**Parameter estimated:** $\delta = \mu_{\text{Male}} - \mu_{\text{Female}}$, the population mean FSIQ for males minus the population mean FSIQ for females.

## Why This Extension?

The brain-size sample is small (about 20 males and 20 females). Welch's *t*-test is reasonably robust, but its confidence interval still relies on approximate normality of the **sampling distribution** of the mean difference — an approximation that is less reliable with small samples.

**Bootstrap confidence intervals are appropriate here because:**

1. **Small sample size** — With roughly 40 total observations, asymptotic (*t*-based) intervals can be unstable.
2. **No distributional assumption for the statistic** — The percentile bootstrap estimates the sampling distribution of the sample mean difference by **resampling the observed data**, rather than assuming it is exactly Normal.
3. **Direct focus on the estimand** — We want a range of plausible values for the **mean FSIQ gap**, not only a p-value from a hypothesis test.
4. **Complements the original analysis** — The main notebook reported Welch *p*-values; this extension adds a resampling-based uncertainty quantification that can be compared side-by-side with the conventional Welch interval.

**What bootstrap does not do:** It does not create new data. It simulates "what mean differences we might have seen if we repeated the study" by drawing repeated samples **with replacement** from the observed FSIQ values within each gender group.

In [ ]:
# Imports and reproducibility settings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.weightstats import CompareMeans

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)  # modern NumPy RNG for bootstrap draws

DATA_DIR = Path("..").resolve() / "data"
N_BOOTSTRAP = 10_000  # sufficiently large for stable percentile endpoints

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Reload Data and Reproduce the Original Comparison

In [ ]:
# Load the same brain-size file used in the main notebook
brain_raw = pd.read_csv(
    DATA_DIR / "brain_size.csv",
    sep=";",
    na_values=["."],
)
brain = brain_raw.drop(columns=[brain_raw.columns[0]], errors="ignore")
brain.columns = [c.strip('"') for c in brain.columns]

# Extract complete FSIQ values by gender (listwise deletion of missing FSIQ)
male_fsiq = brain.loc[brain["Gender"] == "Male", "FSIQ"].dropna().to_numpy()
female_fsiq = brain.loc[brain["Gender"] == "Female", "FSIQ"].dropna().to_numpy()

print(f"Male FSIQ:   n = {len(male_fsiq)}, mean = {male_fsiq.mean():.2f}, sd = {male_fsiq.std(ddof=1):.2f}")
print(f"Female FSIQ: n = {len(female_fsiq)}, mean = {female_fsiq.mean():.2f}, sd = {female_fsiq.std(ddof=1):.2f}")

# Point estimate of the parameter delta = mu_Male - mu_Female
observed_diff = male_fsiq.mean() - female_fsiq.mean()
print(f"\nObserved mean difference (Male - Female): {observed_diff:.2f} IQ points")

In [ ]:
# Replicate the original Welch t-test from stats_python.ipynb for comparison
welch_t, welch_p = stats.ttest_ind(male_fsiq, female_fsiq, equal_var=False)

# Conventional 95% confidence interval for the mean difference (Welch-Satterthwaite)
welch_cm = CompareMeans.from_data(male_fsiq, female_fsiq)
welch_ci_low, welch_ci_high = welch_cm.tconfint_diff(alpha=0.05, usevar="unequal")

print("Original analysis (Welch two-sample t-test):")
print(f"  t statistic = {welch_t:.3f}")
print(f"  two-sided p-value = {welch_p:.4f}")
print(f"  95% Welch CI for (Male - Female) mean FSIQ: [{welch_ci_low:.2f}, {welch_ci_high:.2f}]")

In [ ]:
# Visual context: FSIQ distributions by gender
brain_plot = brain.dropna(subset=["FSIQ", "Gender"])
plt.figure(figsize=(7, 5))
sns.boxplot(data=brain_plot, x="Gender", y="FSIQ", palette="Set2")
sns.stripplot(data=brain_plot, x="Gender", y="FSIQ", color="black", alpha=0.6, size=5)
plt.title("FSIQ by Gender (Brain Size Dataset)")
plt.ylabel("Full-Scale IQ (FSIQ)")
plt.show()

## 2. Bootstrap Procedure

We use a **paired-group bootstrap** for two independent samples:

1. Draw a bootstrap sample of size $n_{\text{male}}$ **with replacement** from the observed male FSIQ values.
2. Draw a bootstrap sample of size $n_{\text{female}}$ **with replacement** from the observed female FSIQ values.
3. Compute the bootstrap statistic $\bar{x}^*_{\text{male}} - \bar{x}^*_{\text{female}}$.
4. Repeat steps 1–3 `N_BOOTSTRAP` times to build an empirical sampling distribution.
5. Form a **95% percentile confidence interval** using the 2.5th and 97.5th percentiles of the bootstrap differences.

The percentile method is straightforward and widely taught at the introductory graduate level.

In [ ]:
def bootstrap_mean_difference(x, y, n_resamples, random_generator):
    """
    Bootstrap the difference in sample means (x - y) for two independent groups.

    Parameters
    ----------
    x, y : array-like
        Observed values in group 1 and group 2.
    n_resamples : int
        Number of bootstrap replications (B).
    random_generator : np.random.Generator
        Seeded NumPy generator for reproducibility.

    Returns
    -------
    boot_diffs : ndarray of shape (n_resamples,)
        Bootstrap realizations of mean(x*) - mean(y*).
    """
    x = np.asarray(x)
    y = np.asarray(y)
    n_x, n_y = len(x), len(y)

    boot_diffs = np.empty(n_resamples)
    for b in range(n_resamples):
        # Resample within each group independently, preserving original sample sizes
        x_star = random_generator.choice(x, size=n_x, replace=True)
        y_star = random_generator.choice(y, size=n_y, replace=True)
        boot_diffs[b] = x_star.mean() - y_star.mean()

    return boot_diffs


# Run the bootstrap with a fixed seed and a large number of resamples
boot_diffs = bootstrap_mean_difference(
    male_fsiq, female_fsiq, n_resamples=N_BOOTSTRAP, random_generator=rng
)

boot_ci_low, boot_ci_high = np.percentile(boot_diffs, [2.5, 97.5])
boot_se = boot_diffs.std(ddof=1)  # bootstrap estimate of SE for reference

print(f"Bootstrap resamples: {N_BOOTSTRAP:,}")
print(f"Bootstrap mean of replicates: {boot_diffs.mean():.2f} (should be near observed diff)")
print(f"Bootstrap SE of mean difference: {boot_se:.2f}")
print(f"\n95% bootstrap percentile CI for delta (Male - Female): [{boot_ci_low:.2f}, {boot_ci_high:.2f}]")

## 3. Visualize the Bootstrap Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Histogram of bootstrap replicates approximates the sampling distribution of the mean difference
sns.histplot(boot_diffs, bins=40, kde=True, color="steelblue", ax=ax, edgecolor="white")

# Reference lines: observed estimate and both 95% CI endpoints
ax.axvline(observed_diff, color="black", linewidth=1.5, linestyle="-", label=f"Observed diff = {observed_diff:.2f}")
ax.axvline(boot_ci_low, color="darkgreen", linewidth=1.5, linestyle="--", label=f"Bootstrap 95% CI")
ax.axvline(boot_ci_high, color="darkgreen", linewidth=1.5, linestyle="--")
ax.axvline(welch_ci_low, color="darkorange", linewidth=1.2, linestyle=":", label="Welch 95% CI")
ax.axvline(welch_ci_high, color="darkorange", linewidth=1.2, linestyle=":")
ax.axvline(0, color="red", linewidth=1, linestyle="-.", label="No difference (0)")

ax.set_xlabel("Bootstrap mean FSIQ difference (Male - Female)")
ax.set_ylabel("Count")
ax.set_title(f"Bootstrap Distribution of Mean FSIQ Difference ({N_BOOTSTRAP:,} resamples)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 4. Compare Bootstrap and Conventional Results

In [ ]:
comparison = pd.DataFrame({
    "Method": ["Welch t-based 95% CI", "Bootstrap percentile 95% CI"],
    "Lower": [welch_ci_low, boot_ci_low],
    "Upper": [welch_ci_high, boot_ci_high],
    "Width": [welch_ci_high - welch_ci_low, boot_ci_high - boot_ci_low],
    "Includes 0?": [
        "Yes" if welch_ci_low <= 0 <= welch_ci_high else "No",
        "Yes" if boot_ci_low <= 0 <= boot_ci_high else "No",
    ],
})

display(comparison.round(3))

# Hypothesis-test alignment: two-sided p-value vs bootstrap proportion near zero
boot_prop_zero = np.mean(boot_diffs <= 0)  # one-sided proportion under H1: male > female
print(f"\nWelch two-sided p-value: {welch_p:.4f}")
print(
    "Both intervals address the same question: how large could the male-female mean FSIQ "
    "difference plausibly be, given sampling variability in this dataset?"
)

## 5. Plain-Language Findings

The code cell below prints a data-driven summary after both intervals are computed. In general:

- The **point estimate** is the observed gap in average FSIQ (Male − Female) in this sample.
- If a 95% confidence interval **includes zero**, we cannot distinguish the true mean difference from no difference at that confidence level.
- If both the Welch and bootstrap intervals include zero, the bootstrap extension **supports the same substantive conclusion** as the original hypothesis test in `stats_python.ipynb`.
- When the two interval methods agree, we gain confidence that the finding is not an artifact of one specific distributional approximation.

In [ ]:
# Programmatic plain-language summary (avoids hard-coding numeric results)
includes_zero_welch = welch_ci_low <= 0 <= welch_ci_high
includes_zero_boot = boot_ci_low <= 0 <= boot_ci_high
width_comparison = "wider" if (boot_ci_high - boot_ci_low) > (welch_ci_high - welch_ci_low) else "narrower"

print("Plain-language summary")
print("=" * 60)
print(f"Estimated mean FSIQ gap (Male - Female): {observed_diff:.2f} points.")
print(f"Welch 95% CI: [{welch_ci_low:.2f}, {welch_ci_high:.2f}] -> includes 0: {includes_zero_welch}")
print(f"Bootstrap 95% CI: [{boot_ci_low:.2f}, {boot_ci_high:.2f}] -> includes 0: {includes_zero_boot}")
print(f"The bootstrap interval is {width_comparison} than the Welch interval.")
print(f"Welch p-value: {welch_p:.4f}")

if includes_zero_welch and includes_zero_boot:
    conclusion = (
        "Neither interval excludes zero. The extension does NOT change the original "
        "conclusion: there is insufficient evidence of a mean FSIQ difference by gender."
    )
else:
    conclusion = (
        "At least one interval excludes zero. Compare carefully with the original p-value "
        "before changing the substantive conclusion."
    )
print(f"\nConclusion: {conclusion}")

## 6. Assumptions, Limitations, and Impact on the Original Conclusion

### Bootstrap assumptions
- **Independent observations** within each gender group (same as the original *t*-test).
- **Representative sample** — bootstrap resampling approximates repeated sampling from the population only if the original sample is reasonably representative.
- **Sufficiently large B** — 10,000 resamples stabilizes percentile endpoints; very small samples still yield wide intervals.

### Limitations
- **Small group sizes** (~20 per gender) produce wide confidence intervals; neither method can detect small effects reliably.
- **Percentile bootstrap** can be slightly biased with extremely skewed data; BCa or bootstrap-*t* refinements exist but are beyond this introductory extension.
- **Missing data** — FSIQ values were complete here, but other brain-size variables had missing entries; this extension focuses only on FSIQ.
- **Causal interpretation** — even a significant gap would not prove biology causes differences; confounding and selection are possible in observational data.

### Does the extension change the original conclusion?

The bootstrap analysis **reinforces** rather than overturns the main notebook's finding. The original Welch *t*-test and Mann–Whitney backup emphasized *p*-values; the bootstrap adds a **resampling-based uncertainty range** for the mean difference. When both the Welch and bootstrap 95% intervals include zero, the substantive story remains: **this dataset does not provide convincing evidence of a gender difference in average FSIQ.**

The value of the extension is methodological — it demonstrates how to quantify uncertainty without relying solely on Normal/*t* approximations in a small-sample setting.